# [GPU 실습] 인터뷰 녹음 → 받아쓰기 → 속보 초안 → 인용 확인
Whisper(음성 인식 모델)를 **내 컴퓨터에서** 실행해요. 녹음 파일이 밖으로 나가지 않아요.

준비: 휴대폰으로 1~2분 모의 인터뷰를 녹음해 `interview.m4a`로 같은 폴더에 (mp3·wav도 OK)
⚠️ 남의 방송·영상 음성은 쓰지 않아요. 녹취록도 온라인에 올리지 않아요.

| 어디서 | 설치 |
|---|---|
| Colab | 런타임 유형 **T4 GPU** → 아래 설치 칸만 실행 |
| 윈도우 (RTX 50 시리즈) | 터미널에서 먼저: `pip install torch --index-url https://download.pytorch.org/whl/cu128` · `winget install ffmpeg` (설치 후 VS Code 다시 열기) |

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
# Colab용 설치 칸. VS Code는 README 순서대로(GPU용 torch 먼저) 설치했다면 건너뛰기
if IN_COLAB:
    get_ipython().run_line_magic('pip', 'install -q openai-whisper')

## 1. 받아쓰기 — GPU면 큰 모델(turbo), CPU면 작은 모델(small)

In [ ]:
import time, torch, whisper
import pandas as pd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL = 'turbo' if DEVICE == 'cuda' else 'small'
AUDIO = 'interview.m4a'

model = whisper.load_model(MODEL, device=DEVICE)          # 처음 한 번 내려받기
t = time.time()
result = model.transcribe(AUDIO, language='ko', fp16=(DEVICE == 'cuda'))
print(f'{DEVICE} · {MODEL} 모델 · {time.time() - t:.0f}초 걸림')

In [ ]:
seg = pd.DataFrame([{'시작': s['start'], '내용': s['text'].strip()} for s in result['segments']])
seg['시각'] = seg['시작'].apply(lambda s: f'{int(s // 60):02d}:{int(s % 60):02d}')
with open('transcript.txt', 'w', encoding='utf-8') as f:
    for _, r in seg.iterrows():
        f.write(f"[{r['시각']}] {r['내용']}\n")
seg[['시각', '내용']]

👀 **먼저 사람이 읽기** — 이름·숫자·고유명사는 잘못 들리기 쉬워요. 틀린 곳은 `transcript.txt`를 직접 고치고 다음으로

## 2. 녹취록으로 속보 초안 (GPT)

In [ ]:
# ▶ API 키 불러오기 — Colab은 🔑 보안 비밀, VS Code는 같은 폴더의 .env 파일
#   .env 파일 내용 예:  OPENAI_API_KEY=sk-...   (GitHub에 올리지 않기)
def get_key(name='OPENAI_API_KEY'):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        print('pip install python-dotenv 후 다시 실행')
    return os.getenv(name)

OPENAI_API_KEY = get_key()
print('API 키:', '준비됨' if OPENAI_API_KEY else '없음 → 키를 등록하세요')

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
transcript = open('transcript.txt', encoding='utf-8').read()

prompt = f"""아래 인터뷰 녹취록으로 속보 3줄을 써.
- 녹취록에 있는 말만 써. 없는 숫자·원인·전망은 쓰지 마.
- 인용은 큰따옴표 안에 녹취록 그대로 쓰고, 뒤에 [시각]을 붙여.

{transcript}"""
r = client.chat.completions.create(model='gpt-4.1-mini', messages=[{'role': 'user', 'content': prompt}], temperature=0)
draft = r.choices[0].message.content.strip()
print(draft)

## 3. 인용 확인 — 따옴표 속 말이 녹취록에 정말 있나?

In [ ]:
import re
def norm(s):
    return re.sub(r'[\s.,!?·]', '', s)

quotes = re.findall(r'["“](.+?)["”]', draft)
for q in quotes:
    print('✅ 녹취록에 있음' if norm(q) in norm(transcript) else '❌ 녹취록에 없음 — 고치기', '|', q)

### 정리
받아쓰기(기계) → **사람이 고치기** → 초안(기계) → **인용 대조(사람+코드)** → 발행 판단(사람)
11/12의 카오스 보도 흐름과 같아요. 받아쓴 글은 n8n 흐름의 입력으로도 쓸 수 있어요.